# **Detector de Conflitos em Textos - PGC UFABC**

Espaço para desenvolvimento, teste e implementação do modelo de detecção de conflitos em textos, proposto no Projeto de Graduação em Computação da Universidade Federal do ABC



**Orientandos:**

*   Caio Cardoso dos Santos
*   Victor Ravazio de Lima

**Orientador:**

*   Prof. Dr. Carlos da Silva dos Santos




# **Conceitos e direções**

## **Conceiros essenciais - Super Resumo**



*   BERT - Modelo de Linguagem baseado em Transformers - Aprende relações semanticas profundas em textos

*   BERTimbau - BERT pré-treinado em portugês Br

*   Tokenizer - converte texto em numeros (IDS), onde cada inteiro é um indice no vocabulo.

  EX: "Eu adorei esse produto" => input_ids = [101, 1198, 17842, 1239, 15323, 102].
  
  Basicamento o Bert possui um dicionario estático e cada valor desse vetor de input corresponde a posição em que determinada palavra ou prefixo está localizada neste dicionario.


*   Fine-tuning - Ajustar o modelo para a tarefa especifica
*   Modelo: [neuralmind/bert-base-portuguese-cased](https://https://huggingface.co/neuralmind/bert-base-portuguese-cased)

**Fonte:** @inproceedings{souza2020bertimbau,
  author    = {F{\'a}bio Souza and
               Rodrigo Nogueira and
               Roberto Lotufo},
  title     = {{BERT}imbau: pretrained {BERT} models for {B}razilian {P}ortuguese},
  booktitle = {9th Brazilian Conference on Intelligent Systems, {BRACIS}, Rio Grande do Sul, Brazil, October 20-23 (to appear)},
  year      = {2020}
}

Fontes auxiliares e projetos:

Fonte 2: https://arthuremanuel-carosia.medium.com/usando-o-bert-para-an%C3%A1lise-de-sentimentos-363f76eee15e

Fonte 3: https://gist.github.com/Hugobsan/b507b76c7b5fc470d77b3a924211430d#file-pr-processamento-ipynb




##**Pipeline idealizado (provisório?)**


Dado o Dataset com Tweets:

*   Cada Dataset temático é divido em 3 subestruturas: Tweets genéricos, repys e quotes


  
*   Gera-se pares de tweets dentro de uma mesma subestrutura: Reply com seu respectivo replied tweet; Quote com seu respsctivo Quoted tweet; e original pareado com outro tweet origginal de forma aleatoria (amostragem aleatoria)

*   Para cada par de frases de cada grupo, constrói um dataset com essas informações: frase_1 - frase_2 - Classe - conflito (0/1)
*   Esta tabela seria o dataset de entrada para o bert - Este então aprende os padrões textuais que geralmente associam conflitos - “Que tipo de relação semântica entre frase_1 e frase_2 gera um situação de conflito 1?” -  Ex: bom x ruim; ótimo x péssimo

*   O modelo no fim é necessário pois ele aprende uma fronteira suave e não discreta.






# **Preparação do ambiente**


## **Validando GPU**

In [31]:
!nvidia-smi #verificando se o ambiente está configurado para GPU


Mon May 18 13:49:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## **Instalando bibliotecas**

**Sobre as bibliotecas:**


*   Transformers (coleção de modelos pré-treinados em Transformers)
*   datasets (fornece alguns datasets otimizados prontos)
*   accelerate (gerencia multi gpu/otimiza treino em gpu)
*   evaluate (biblioteca de metricas padronizadas)

*   PyTorch (framework de deep learning)

* As 4 primeiras pertencem ao Hugging Face, espécie de "GitHub" de modelos IA







In [32]:
#Instalando:
!pip install -q transformers datasets torch accelerate evaluate

In [33]:
# modulo para importar dataset a partir de um link
!pip install gdown

# **Importações**

## **Importando Bibliotecas**

**Sobre as bibliotecas:**


*  BertTokenizer(Tokenizar textos)
*   BertModel (Bert puro)
*   BertForSequenceClassification (BERT + Camada de classificador)
*   Trainer (modulo para treino)

*   TrainigArguments (modulo para treino)

*   Gdown (baixar arquivos direto do drive)

* Dataset (otimiza e facilita a formatação do datataset para treino no Bert)

* sklearn (biblioteca que fornece ferramentas prontas para realizar tarefas de ML, como por exemplo "train_test_split" que divide automaticamente treino e teste, ou as metricas de avaliação que abstrai os calculos de precisão, recall e etc)






In [34]:
import torch
import numpy as np
import pandas as pd
import gdown

from transformers import (
    BertTokenizer,
    BertModel,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score


## **Importando o Modelo**

In [35]:
#Tokenizer
tokenizer = BertTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

#Modelo base:
model = BertForSequenceClassification.from_pretrained("neuralmind/bert-base-portuguese-cased", num_labels = 2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

## **Importando o dataset**

In [36]:
# baixando dataset para o collab
file_id = "1eGuTbLr6W5crwM5PKlDt8Rab1xNRUYnX"
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "cloroquina_replys.csv", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1eGuTbLr6W5crwM5PKlDt8Rab1xNRUYnX
To: /content/cloroquina_replys.csv
100%|██████████| 2.37M/2.37M [00:00<00:00, 96.5MB/s]


'cloroquina_replys.csv'

In [37]:
#carregando o dataset
df = pd.read_csv ("cloroquina_replys.csv")
df = df.iloc[:100]
df = df[df['Conflict'] != -1] #eliminando indeterminados
df.shape

(100, 4)

In [38]:
#Verificando distribuição das classes
print("Quantidade de anotações por classes:")
print("0 - Não Conflito: ", df[df["Conflict"] == 0].shape[0])
print("1 - Conflito: ", df[df["Conflict"] == 1].shape[0])

Quantidade de anotações por classes:
0 - Não Conflito:  56
1 - Conflito:  44


# **Modelo - Conflict Detector**

## **Separando treino e teste**

In [39]:
#Separando dados em treino e teste
# train_test_split é um método da biblioteca sklearn que abstrai a divisão de treino e test
# O parametro stratify mantém a proporção das classes - Originalmente 80% não conflito e 20% não conflito, assim também será em train e test.
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["Conflict"])

#convertendo para o formato "datasets" do Hugging face - Abstrai alguns procedimentos, como formatar para tensores e etc
train_dataset = Dataset.from_pandas(train_df, preserve_index=False) #preserve index = false - não inclui os indices do df original
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)


In [40]:
train_dataset

Dataset({
    features: ['tweet_1', 'tweet_2', 'Class', 'Conflict'],
    num_rows: 80
})

## **Tokenização**

Tokenização consiste em transformar o  texto bruto em uma representação numérica estruturada que o modelo bert consiga processsar. Para tal, o texto bruto é quebrado em subpalavras, e em seguida é produzido um vetor onde cada Token (subpalavra) é convertida em Ids númericos correspondentes a indices do vocabulário aprendido pelo modelo.

O processo de tokenização gera os seguintes vetores:

*   "Input_ID": Id de cada token

*   "token_type_ids": Diz qual token pertence a qual sentença (Bert pode receber pares de sentenças)

*   "attention_mask": Indica quais posições do vetor são reais e quais são padding


Para a tokenização adotada no trabalho, a ideia é aproveitar a estrutura do Bert para Tokenizar os pares de tweets em conjunto. Desta forma, o embendding produzido considera os textos em pares e não isoladamente, e isso garante que o modelo aprenda interações profundas entre as palavras de cada frase.

Para a estrutura do Bert, temos o seguinte formato de entrada:

[CLS] Texto_1 [sep] Texto_2 [sep]

Desta forma, para alimentar o modelo com este formato de entrada, precisamos chamar a função "tokenizer" e passar como parametros os 2 textos e infos de controle como um tamanho de padding (garantir sentenças do mesmo tamanho) e que sejam truncadas

Estes vetores tokenizados serão a entrada do modelo para a produção dos embenddings contextualizados.

Obs: O padding é importante para padronizar o tamanho das amostras. Como o processamento é realizado em batches (varias amostras processadas simultaneamente), é necessário que todas formem elementos de mesmo tamanho, é o formato que o modelo espera receber.

In [41]:
#Função de tokenização
def tokenize_function(pair):

    return tokenizer(
        pair["tweet_1"],
        pair["tweet_2"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [42]:
#Aplicando a Tokenização -
# "map" é um método da biblioteca "datasets" do Hugging Face que basicamente aplica tokenize_function em cada entrada do df
# Naturalmente "train_dataset" e "test_dataset" é um objeto "Dataset" (da biblioteca do hugging face)
train_dataset = train_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [43]:
#Formato pós tokenização
train_dataset

Dataset({
    features: ['tweet_1', 'tweet_2', 'Class', 'Conflict', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 80
})

In [44]:
#Exemplo
print(train_dataset[0])

{'tweet_1': 'O protocolo da cloroquina é o "foda-se" institucionalizado.', 'tweet_2': 'O que fazer se o médico entender que aquele paciente não tem indicação de cloroquina? Prescreve mesmo assim?', 'Class': 1, 'Conflict': 0.0, 'input_ids': [101, 231, 14491, 180, 20139, 11198, 253, 146, 107, 227, 285, 118, 176, 107, 19237, 2303, 119, 102, 231, 179, 1434, 176, 146, 4714, 9050, 179, 6086, 10268, 346, 376, 9403, 125, 20139, 11198, 136, 2311, 1128, 301, 653, 1016, 136, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,